In [3]:
#%pip install notion_client
#%pip install openai

In [476]:
import os
from openai import OpenAI
from notion_client import Client
from dotenv import load_dotenv
import ast

# From .env get notion_token
load_dotenv()
notion_token = os.getenv('NOTION_TOKEN')
notion = Client(auth=notion_token)

# Create a dict object to map categorydisplay to different language
categorydisplay_dict = {
    "일상기록": {"en": "Daily Life", "es": "Vida Diaria"},
    "의료정보학": {"en": "Clinical Informatics", "es": "Informática Clínica"},
    "연구일기": {"en": "Research", "es": "Investigación"},
    "공부일기": {"en": "Learning", "es": "Aprendizaje"},
}

In [477]:
import re
import os

def extract_markdown(blocks):
    """
    Extract Markdown content from Notion blocks.

    Parameters:
    blocks (list): List of Notion blocks.

    Returns:
    str: Markdown content.
    """
    markdown_lines = []

    for block in blocks['results']:
        block_type = block['type']

        block_content = block[block_type]
        text =''

        if block_type != 'image' and block_type != 'divider':

            for rt in block_content['rich_text']:
    
                #Check if this text is linked
                if rt['text']['link'] != None:
                    tmp = f"[{rt['text']['content']}]({rt['text']['link']['url']})"
                else:
                    tmp = f"{rt['text']['content']}"

                #Check if annotations says if text is bold/italic/strikethrough/underline/code/colored
                if rt['annotations']['bold'] == True:
                    tmp = f"**{tmp}**"
                if rt['annotations']['italic'] == True:
                    tmp = f"*{tmp}*"
                if rt['annotations']['strikethrough'] == True:
                    tmp = f"~~{tmp}~~"
                if rt['annotations']['underline'] == True:
                    tmp = f"<ins>{tmp}</ins>"
                if rt['annotations']['code'] == True:
                    tmp = f"`{tmp}`"
                if rt['annotations']['color'] != 'default':
                    tmp = f"<span style='color:{rt['annotations']['color']}'>{tmp}</span>"

                text += tmp

            # Add two spaces at the end of each line to create line breaks
            text += '  '

        if block_type == 'paragraph':
            markdown_lines.append(text)
        elif block_type == 'heading_1':
            markdown_lines.append(f"# {text}")
        elif block_type == 'heading_2':
            markdown_lines.append(f"## {text}")
        elif block_type == 'heading_3':
            markdown_lines.append(f"### {text}")
        elif block_type == 'bulleted_list_item':
            markdown_lines.append(f"- {text}")
        elif block_type == 'numbered_list_item':
            markdown_lines.append(f"1. {text}")
        elif block_type == 'to_do':
            checked = block_content['checked']
            markdown_lines.append(f"- [{'x' if checked else ' '}] {text}")
        elif block_type == 'quote':
            text = text.replace('\n', '\n> ')  # Add > at the beginning of each line
            markdown_lines.append(f"> {text}")
        elif block_type == 'code':
            language = block_content['language']
            markdown_lines.append(f"```{language}\n{text}\n```")
        elif block_type == 'callout':
            icon = block_content['icon']['emoji']
            markdown_lines.append(f"> {icon} {text}")
        elif block_type == 'divider':
            markdown_lines.append("---")
        elif block_type == 'image':
            # Suppose we only use external url for images... for convenience
            url = block_content['external']['url']
            caption = block_content['caption'][0]['plain_text']
            markdown_lines.append(f"![{caption}]({url})  ")

    return markdown_lines

def extract_notion_page_id(notion_url):
    """
    Extract the Notion page ID from a Notion URL.
    
    Parameters:
    notion_url (str): The URL of the Notion page.
    
    Returns:
    str: The extracted Notion page ID.
    """
    # Define the regular expression pattern to match the Notion page ID - geting 32-character string
    pattern = re.compile(r'([a-f0-9]{32})')
    
    # Search for the pattern in the Notion URL
    match = pattern.search(notion_url)
    
    if match:
        # Extract the page ID
        page_id = match.group(1)
        
        # Insert hyphens in the pattern 8-4-4-4-12
        formatted_page_id = f"{page_id[:8]}-{page_id[8:12]}-{page_id[12:16]}-{page_id[16:20]}-{page_id[20:]}"
        
        return formatted_page_id
    else:
        raise ValueError("Invalid Notion URL or page ID not found.")
    

# Create a code to get properties from Notion Page to formulate Front Matter for Jekyll
def extract_frontmatter(page_id):
    """
    Get the properties of a Notion page.
    
    Parameters:
    page_id (str): The ID of the Notion page.
    
    Returns:
    dict: The properties of the Notion page.

    * of note, for my Jekyll page, I used the following properties
        - title: title of post
        - date: date post is created. Instead of using date in Notion, I used the date that I manually input in Notion
        - tags: tags of the post
        - categories: categories of the post
        - categorydisplay: wierd name, I know. But this is for display purposes in Jekyll
        - lang: language of the post; either kr, en, or es; default is kr
        - thumbnail: thumbnail image of the post
        - subtitle: subtitle of the post
    """
    # Get the page data
    page_data = notion.pages.retrieve(page_id)
    
    # Get the title of the page
    title = page_data['properties']['title']['rich_text'][0]['plain_text']
    # Get the date of the page
    date = page_data['properties']['date']['date']['start']
    # Get the tags of the page
    tags = [ i['name'] for i in page_data['properties']['tags']['multi_select']]
    # get the categories of the page
    categories = page_data['properties']['categories']['select']['name']
    # Get the categorydisplay of the page
    categorydisplay = page_data['properties']['categorydisplay']['select']['name']
    # Get the lang of the page
    lang = page_data['properties']['lang']['select']['name']
    # Get the thumbnail of the page
    thumbnail = page_data['properties']['thumbnail']['files'][0]['name']
    # Get the subtitle of the page
    subtitle = page_data['properties']['subtitle']['rich_text'][0]['plain_text']
    
    # Create a dictionary of the properties
    properties = {
        'title': title,
        'date': date,
        'tags': ' '.join(tags),
        'categories': categories,
        'categorydisplay': categorydisplay,
        'lang': lang,
        'thumbnail': thumbnail,
        'subtitle': subtitle
    }
    
    return properties

# Write Jekyll post Markdown file
def write_jekyll_post_from_notion_page(page_id):
    """
    Write a Jekyll post Markdown file.
    
    Parameters:
    page_id (str): The ID of the Notion page.

    """
    # Get the title of the page
    page_fm = extract_frontmatter(page_id)
    page_md = extract_markdown(notion.blocks.children.list(page_id))

    # Define the filename of the Markdown file
    filename = f"{page_fm['date']}-{page_fm['title']}.md"

    # Define the front matter of the Markdown file
    front_matter = f"""---
layout: post
permalink: /:title/
title: "{page_fm['title']}"
date: {page_fm['date']} 00:00:00 -0400
tags: {page_fm['tags']}
categories: {page_fm['categories']}
categorydisplay: {page_fm['categorydisplay']}
lang: {page_fm['lang']}
thumbnail: {page_fm['thumbnail']}
subtitle: {page_fm['subtitle']}
---\n"""

    # Write the Markdown content to the file
    # Write in directory ./_posts/{lang}/{categories}/{filename}
    with open(os.path.join('_posts', page_fm['lang'] , page_fm['categories'],filename), 'w') as file:
        file.write(front_matter)
        for line in page_md:
            file.write(f"{line}\n\n")
    
    # Will return page_fm and filename for reference
    print(f"Jekyll post Markdown file written: {os.path.join('_posts', page_fm['lang'] , page_fm['categories'],filename)}")
    return page_fm, page_md, filename

# Will Create a Code to translate pmd to English using OpenAI

# Define the OpenAI API key
client = OpenAI(
  api_key=os.getenv('OPENAI_API_KEY'),
)

def translate_markdown(markdown_string, frontmatter_dict, target_language):
    """
    Translate markdown content to the target language using OpenAI.

    Parameters:
    pmd (list): List of markdown strings.
    target_language (str): Target language for translation.

    Returns:
    str: Translated markdown content.
    """
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a translator of a Jekyll blog. Your job is to translate contents in markdown into given language; keep all markdown syntax."},
            {"role": "user", "content": f"Translate the following Korean markdown page to {target_language}: {markdown_string}"}
        ]
    )

    title_completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Translate following Korean phrase to given language."},
            {"role": "user", "content": f"Translate this to {target_language}: {frontmatter_dict['title']}"}
        ]
    )

    subtitle_completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Translate following Korean phrase to given language."},
            {"role": "user", "content": f"Translate this to {target_language}: {frontmatter_dict['subtitle']}"}
        ]
    )

    return completion.choices[0].message, title_completion.choices[0].message, subtitle_completion.choices[0].message

# Execute

- Define notion page url
- Create root markdown file
- Use the file to create Spanish/English pages
- Store them accordingly

In [353]:
page_url = "https://seungwooklee.notion.site/b788df8a2aff448bbcb61aee55fd52c8?pvs=4"

# Example usage
#notion_url = "https://www.notion.so/your-page-title-a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6"
page_id = extract_notion_page_id(page_url)
print("Notion Page ID:", page_id)

pfm, pmd, fn = write_jekyll_post_from_notion_page(page_id)

# Translate the markdown content to English
translated_content_en, en_title, en_subtitle = translate_markdown(' '.join(pmd), pfm, 'English')

en_md_list = translated_content_en.content.split('\n')

filename = f"{pfm['date']}-{pfm['title']}.md"

# Define the front matter of the Markdown file
front_matter = f"""---
layout: post
permalink: /en/:title/
title: "{en_title.content}"
date: {pfm['date']} 00:00:00 -0400
tags: {pfm['tags']}
categories: {pfm['categories']}
categorydisplay: {categorydisplay_dict[pfm['categorydisplay']]['en']}
lang: en    
thumbnail: {pfm['thumbnail']}
subtitle: {en_subtitle.content}
---\n"""

with open(os.path.join('_posts', 'en', pfm['categories'], filename), 'w') as file:
    file.write(front_matter)
    for line in en_md_list:
        file.write(f"{line}\n\n")

# Will return pfm and filename for reference
print(f"Jekyll post Markdown file written: {os.path.join('_posts', 'en' , pfm['categories'], filename)}")


# Translate the markdown content to Spanish
translated_content_es, es_title, es_subtitle = translate_markdown(' '.join(pmd), pfm, 'Spanish')

es_md_list = translated_content_es.content.split('\n')

filename = f"{pfm['date']}-{pfm['title']}.md"

# Define the front matter of the Markdown file
front_matter = f"""---
layout: post
permalink: /es/:title/
title: "{es_title.content}"
date: {pfm['date']} 00:00:00 -0400
tags: {pfm['tags']}
categories: {pfm['categories']}
categorydisplay: {categorydisplay_dict[pfm['categorydisplay']]['es']}
lang: es    
thumbnail: {pfm['thumbnail']}
subtitle: {es_subtitle.content}
---\n"""

with open(os.path.join('_posts', 'es', pfm['categories'], filename), 'w') as file:
    file.write(front_matter)
    for line in es_md_list:
        file.write(f"{line}\n\n")

# Will return pfm and filename for reference
print(f"Jekyll post Markdown file written: {os.path.join('_posts', 'es' , pfm['categories'], filename)}")

Notion Page ID: b788df8a-2aff-448b-bcb6-1aee55fd52c8
Jekyll post Markdown file written: _posts\kr\life\2024-07-02-파이프라인 설정 실험.md
